# TimesFM 2.5 Training - Git Version

**Clone from GitHub, Train on T4 GPU, Push Results Back**

---

## Why This Works:

- ✅ **No manual uploads** - Clone everything from Git
- ✅ **Version control** - Track all changes
- ✅ **Auto-sync** - Push trained models back to GitHub
- ✅ **T4 GPU (16GB)** - Sufficient memory for TimesFM 2.5

## Expected Results:
- **Training time:** ~3 hours
- **R²:** 0.5-0.6 (explains 50-60% variance)
- **QLIKE:** < 1.0 (good volatility forecast)

## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers peft torch pandas numpy pyyaml accelerate

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Clone Repository from GitHub

In [ ]:
# Replace with YOUR GitHub username and repo name
GITHUB_USERNAME = "YOUR_USERNAME"  # @param {type:"string"}
REPO_NAME = "stockvoli-research"  # @param {type:"string"}

# Clone repository
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

# Change to repo directory
import os
os.chdir(f"{REPO_NAME}")

print(f"Working directory: {os.getcwd()}")
print("\nFiles in repository:")
!ls -la

## Step 4: Verify Data is Available

In [ ]:
from pathlib import Path

processed_dir = Path('data/processed')
if processed_dir.exists():
    files = list(processed_dir.glob('*_processed.csv'))
    print(f"✅ Found {len(files)} processed stock files")
    print("\nStock files:")
    for f in sorted(files)[:5]:
        print(f"  - {f.name}")
    print(f"  ... and {len(files)-5} more")
else:
    print(f"❌ Processed data directory not found!")
    print(f"Expected: {processed_dir}")

## Step 5: Setup Git Identity (for pushing results)

In [ ]:
!git config user.email "you@example.com"  # @param {type:"string"}
!git config user.name "Your Name"  # @param {type:"string"}

# Setup GitHub authentication (use PAT for security)
# Create PAT at: https://github.com/settings/tokens
GITHUB_PAT = "YOUR_GITHUB_PAT"  # @param {type:"string"}

# Setup remote with PAT
!git remote set-url origin https://{GITHUB_PAT}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git

print("✅ Git configured for pushing results")

## Step 6: Start Training (T4 GPU - 16GB)

In [ ]:
# Set CUDA memory management
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Run training script
!python src/model_training_fixed.py

## Step 7: Monitor Training Progress

In [ ]:
# Monitor GPU usage
!nvidia-smi -l 2  # Update every 2 seconds

# Check training logs
!tail -f experiments/model_training.log

## Step 8: Push Results Back to GitHub

In [ ]:
# Add trained models and results
!git add models/ experiments/

# Commit results
!git commit -m "feat: Add trained TimesFM 2.5 model

- Training completed on Google Colab T4 GPU
- R²: [ADD YOUR R² HERE]
- QLIKE: [ADD YOUR QLIKE HERE]
- RMSE: [ADD YOUR RMSE HERE]

Co-Authored-By: Claude Sonnet 4.6 <noreply@anthropic.com>"

# Push to GitHub
!git push origin master

print("✅ Results pushed to GitHub!")

## 🆘 Troubleshooting

### "Repository not found"
**Solution:** Make sure you created the repo on GitHub first

### "Authentication failed"  
**Solution:** Create GitHub PAT at https://github.com/settings/tokens

### "Data files not found"
**Solution:** Make sure you pushed the data to GitHub first (see setup guide)

### "CUDA out of memory"
**Solution:** This should NOT happen on T4 16GB. Restart runtime if it does.